In [8]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('BigD_OurD').getOrCreate()
sc = spark.sparkContext

In [9]:
def keepNameGenreSales(line):
    out = line.split(",")
    to_remove = [0, 2, 3, 5]
    for i in sorted(to_remove, reverse=True):
        out.pop(i)
    return out

game_rdd = sc.textFile("vgsales.csv").map(keepNameGenreSales)
print(game_rdd.collect())

[['Name', 'Genre', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'], ['Wii Sports', 'Sports', '41.49', '29.02', '3.77', '8.46', '82.74'], ['Super Mario Bros.', 'Platform', '29.08', '3.58', '6.81', '0.77', '40.24'], ['Mario Kart Wii', 'Racing', '15.85', '12.88', '3.79', '3.31', '35.82'], ['Wii Sports Resort', 'Sports', '15.75', '11.01', '3.28', '2.96', '33'], ['Pokemon Red/Pokemon Blue', 'Role-Playing', '11.27', '8.89', '10.22', '1', '31.37'], ['Tetris', 'Puzzle', '23.2', '2.26', '4.22', '0.58', '30.26'], ['New Super Mario Bros.', 'Platform', '11.38', '9.23', '6.5', '2.9', '30.01'], ['Wii Play', 'Misc', '14.03', '9.2', '2.93', '2.85', '29.02'], ['New Super Mario Bros. Wii', 'Platform', '14.59', '7.06', '4.7', '2.26', '28.62'], ['Duck Hunt', 'Shooter', '26.93', '0.63', '0.28', '0.47', '28.31'], ['Nintendogs', 'Simulation', '9.07', '11', '1.93', '2.75', '24.76'], ['Mario Kart DS', 'Racing', '9.81', '7.57', '4.13', '1.92', '23.42'], ['Pokemon Gold/Pokemon Silver', 'Role-P

In [20]:
from pyspark.sql.functions import sum as _sum, desc


platform_total_sales = df_cleaned.groupBy('Platform') \
    .agg(_sum('Global_Sales').alias('Total_Global_Sales'))

# rp
platform_total_sales = platform_total_sales.repartitionByRange(4, 'Total_Global_Sales')

# sort by most sales most to least top 10
platform_total_sales = platform_total_sales.orderBy(desc('Total_Global_Sales'))

platform_total_sales.show(10)

top_platform = platform_total_sales.first()
print(f"The platform with the most global sales is {top_platform['Platform']} with {top_platform['Total_Global_Sales']:.2f} million units.")
print(f"Number of partitions: {platform_total_sales.rdd.getNumPartitions()}")

+--------+------------------+
|Platform|Total_Global_Sales|
+--------+------------------+
|     PS2|1233.4599999999837|
|    X360| 969.6099999999993|
|     PS3| 949.3499999999987|
|     Wii| 909.8099999999976|
|      DS| 818.9599999999875|
|      PS| 727.3899999999971|
|     GBA| 313.5599999999981|
|     PSP| 291.7099999999948|
|     PS4| 278.0999999999994|
|      PC|255.05000000000103|
+--------+------------------+
only showing top 10 rows
The platform with the most global sales is PS2 with 1233.46 million units.
Number of partitions: 1
